# Evaluación por ejecución y análisis de errores

Proyecto 3 · Deep Learning y Sistemas Inteligentes · Sección 30

## Objetivo

Comparar baseline, QLoRA `r=16`, QLoRA `r=32` y el experimento de reparación sobre los mismos 2 esquemas inéditos y 40 ejemplos.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Ejecuta el notebook desde la raíz o notebooks/."
ARTIFACTS = Path(os.environ.get("PROJECT3_ARTIFACTS", ROOT.parent / "artifacts_locales"))
SEED = 2026
print({"root": str(ROOT), "artifacts": str(ARTIFACTS), "seed": SEED})

{'root': 'C:\\Users\\Daniel\\Desktop\\Proyecto-3_Deep\\Proyecto-3_Grupo-1_DLYSI_Sec-30', 'artifacts': 'C:\\Users\\Daniel\\Desktop\\Proyecto-3_Deep\\artifacts_locales', 'seed': 2026}


## 1. Evaluación opcional

Las cinco corridas ya están versionadas. Active la bandera solo si desea repetir inferencia completa con los adapters locales.

In [2]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    raise RuntimeError('Use scripts/run_validation.py con las rutas de adapters descritas en README.md.')
print('Se usan las predicciones CSV/JSON versionadas; no se repitió inferencia.')

Se usan las predicciones CSV/JSON versionadas; no se repitió inferencia.


## 2. Comparación de corridas

In [3]:
import pandas as pd

run_names = ('baseline', 'qlora_r16', 'qlora_r32', 'ganador_direct', 'ganador_repair')
runs = {name: json.loads((ROOT / f'resultados/corridas/{name}.json').read_text(encoding='utf-8')) for name in run_names}
pd.DataFrame([
    {'corrida': name, 'EX ponderada': runs[name]['weighted_ex'], 'ejecutable': runs[name]['executable_rate'], 'p50 ms': runs[name]['latency_p50_ms'], 'p95 ms': runs[name]['latency_p95_ms']}
    for name in run_names
])

,corrida,EX ponderada,ejecutable,p50 ms,p95 ms
0,baseline,0.038462,0.625,3531.3081,6656.3023
1,qlora_r16,0.974359,0.975,8189.1889,15718.6631
2,qlora_r32,0.974359,0.975,7916.7703,13197.0874
3,ganador_direct,0.974359,0.975,7738.2703,12679.7624
4,ganador_repair,0.974359,0.975,7756.5143,15635.2862


## 3. Error del ganador y efecto de reparación

In [4]:
direct = runs['ganador_direct']
repair = runs['ganador_repair']
{
    'EX_directa': direct['weighted_ex'],
    'EX_reparacion': repair['weighted_ex'],
    'delta_EX_pp': round(100 * (repair['weighted_ex'] - direct['weighted_ex']), 1),
    'errores_directos': {k: v for k, v in direct['error_counts'].items() if k != 'correcto'},
}

{'EX_directa': 0.9743589743589743,
 'EX_reparacion': 0.9743589743589743,
 'delta_EX_pp': 0.0,
 'errores_directos': {'entidad_inexistente': 1}}

## 4. Predicción de la evaluación oculta

In [5]:
summary = json.loads((ROOT / 'resultados/resumen.json').read_text(encoding='utf-8'))
{
    'adapter': summary['selected_adapter'],
    'EX_oculta_estimada': summary['hidden_ex_prediction'],
    'intervalo_estimado': summary['hidden_ex_prediction_interval'],
    'metodo': summary['hidden_ex_prediction_method'],
}

{'adapter': 'qlora_r32',
 'EX_oculta_estimada': 0.874,
 'intervalo_estimado': [0.817, 0.9],
 'metodo': 'Bootstrap determinista de 20,000 remuestras por pregunta con semilla 2026; al intervalo y al punto local se restan 0.10 por cambio de dominio.'}

## Resultado

El baseline obtuvo 3.8% de EX. Ambos adapters obtuvieron 97.4% (39/40). `r=32` ganó por menor p95. La reparación mantuvo la EX y añadió latencia; el único fallo del ganador fue una entidad inexistente.